In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
driver = wb.Chrome()
driver.get("https://datalab.visitkorea.or.kr/datalab/portal/loc/getTourDataFormBeta.do")


# <select> 요소 찾기
select_element = driver.find_element(By.ID, "region1")

# Selenium의 Select 클래스를 사용하여 <select> 요소와 상호작용
select = Select(select_element)

# "전체"라는 텍스트를 가진 옵션 선택
select.select_by_visible_text("전체")

# 검색 버튼이 활성화될 때까지 대기
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'input.btn-search'))
)
search_button.click()

# 검색 버튼 클릭
search_button = driver.find_element(By.CSS_SELECTOR, 'input.btn-search')
search_button.click()

time.sleep(2)

html = driver.page_source

from bs4 import BeautifulSoup as bs
soup = bs(html, 'lxml')

# <tbody> 요소 찾기
tbody = soup.find('tbody', id='areaTXNList')

# 테이블의 모든 행 가져오기
rows = tbody.find_all('tr')

# 각 행의 데이터 출력
for row in rows:
    cells = row.find_all('td')
    cell_texts = [cell.get_text(strip=True) for cell in cells]
    print(cell_texts)


    

# 드라이버 종료
driver.quit()


In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager
import time
from bs4 import BeautifulSoup as bs

# 웹 드라이버 설정 및 페이지 열기
driver = wb.Chrome()
driver.get("https://datalab.visitkorea.or.kr/datalab/portal/loc/getTourDataFormBeta.do")

# <select> 요소 찾기 및 옵션 선택
select_element = driver.find_element(By.ID, "region1")
select = Select(select_element)
select.select_by_visible_text("전체")

# 검색 버튼 클릭
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'input.btn-search'))
)
search_button.click()

time.sleep(2)

def get_page_data(driver):
    """현재 페이지에서 데이터를 가져오는 함수"""
    html = driver.page_source
    soup = bs(html, 'lxml')
    
    tbody = soup.find('tbody', id='areaTXNList')
    if tbody:
        rows = tbody.find_all('tr')
        data = []
        for row in rows:
            cells = row.find_all('td')
            cell_texts = [cell.get_text(strip=True) for cell in cells]
            data.append(cell_texts)
        return data
    return []

# 페이지 번호 변수 초기화
current_page = 1
max_page = 716

while current_page <= max_page:
    # 현재 페이지 데이터 크롤링
    page_data = get_page_data(driver)
    for row in page_data:
        print(row)
    
    if current_page <= 10:
        # 1~10페이지일 때: 직접 페이지 클릭
        next_page_id = f"page{current_page + 1}"
        try:
            next_page = driver.find_element(By.ID, next_page_id)
            next_page.click()
            time.sleep(2)
            current_page += 1
        except Exception as e:
            print(f"오류 발생: {e}")
            break
    else:
        # 11페이지 이후: '다음' 버튼 클릭
        try:
            # '다음' 버튼 클릭하여 다음 페이지로 이동
            next_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.table-forward'))
            )
            next_button.click()
            time.sleep(2)
            current_page += 1  # 다음 페이지로 이동 후 current_page 증가
        except Exception as e:
            print(f"오류 발생: {e}")
            break

    # 마지막 페이지에 도달했는지 확인
    if current_page > max_page:
        break

# 드라이버 종료
driver.quit()


In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager
import time
from bs4 import BeautifulSoup as bs
import pandas as pd

# 웹 드라이버 설정 및 페이지 열기
driver = wb.Chrome()
driver.get("https://datalab.visitkorea.or.kr/datalab/portal/loc/getTourDataFormBeta.do")

# <select> 요소 찾기 및 옵션 선택
select_element = driver.find_element(By.ID, "region1")
select = Select(select_element)
select.select_by_visible_text("전체")

# 검색 버튼 클릭
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'input.btn-search'))
)
search_button.click()

time.sleep(3)

def get_page_data(driver):
    """현재 페이지에서 데이터를 가져오는 함수"""
    html = driver.page_source
    soup = bs(html, 'lxml')
    
    tbody = soup.find('tbody', id='areaTXNList')
    if tbody:
        rows = tbody.find_all('tr')
        data = []
        for row in rows:
            cells = row.find_all('td')
            cell_texts = [cell.get_text(strip=True) for cell in cells]
            data.append(cell_texts)
        return data
    return []

def click_next_button(driver):
    """다음 페이지 버튼 클릭 함수"""
    try:
        next_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.table-forward'))
        )
        next_button.click()
        time.sleep(2)  # 페이지 로딩 대기
    except Exception as e:
        print(f"Error clicking next button: {e}")

def process_page_range(start_page, end_page):
    """주어진 페이지 범위에서 데이터 추출 및 저장"""
    all_data = []
    for page in range(start_page, end_page + 1):
        # 데이터 추출
        page_data = get_page_data(driver)
        all_data.extend(page_data)
        
        # 페이지 데이터 출력
        if page_data:
            for row in page_data:
                print(row)
        
        # 페이지 네비게이션
        if page < end_page:
            try:
                page_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.ID, f"page{page + 1}"))
                )
                page_button.click()
                time.sleep(2)  # 페이지 로딩 대기
            except Exception as e:
                print(f"Error navigating to page {page}: {e}")

    return all_data

def main():
    all_data = []
    
    total_pages = 716
    page_range = 10

    current_page = 1
    while current_page <= total_pages:
        end_page = min(current_page + page_range - 1, total_pages)
        all_data.extend(process_page_range(current_page, end_page))
        
        # "다음" 버튼 클릭하여 다음 페이지 범위로 이동
        if end_page < total_pages:
            click_next_button(driver)
        
        # 현재 페이지 업데이트
        current_page = end_page + 1

    # 데이터프레임으로 변환
    # 데이터의 열 수를 동적으로 결정하기 위해 첫 번째 데이터의 열 수를 확인합니다.
    if all_data:
        num_columns = len(all_data[0])  # 첫 번째 항목의 열 수를 가져옵니다.
        columns = [f"Column{i+1}" for i in range(num_columns)]  # 열 이름을 설정합니다.
        df = pd.DataFrame(all_data, columns=columns)
    else:
        df = pd.DataFrame()  # 데이터가 없는 경우 빈 데이터프레임을 생성합니다.

    # 데이터프레임을 CSV 파일로 저장
    df.to_csv('crawled_data.csv', index=False)

    print("Data saved to crawled_data.csv")
    driver.quit()

if __name__ == "__main__":
    main()

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
from bs4 import BeautifulSoup as bs

# 웹 드라이버 설정 및 페이지 열기
driver = wb.Chrome()
driver.get("https://datalab.visitkorea.or.kr/datalab/portal/loc/getTourDataFormBeta.do")

# <select> 요소 찾기 및 옵션 선택
select_element = driver.find_element(By.ID, "region1")
select = Select(select_element)
select.select_by_visible_text("전체")

# 검색 버튼 클릭
search_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, 'input.btn-search'))
)
search_button.click()

time.sleep(2)

def get_page_data(driver):
    """현재 페이지에서 데이터를 가져오는 함수"""
    html = driver.page_source
    soup = bs(html, 'lxml')
    
    tbody = soup.find('tbody', id='areaTXNList')
    if tbody:
        rows = tbody.find_all('tr')
        data = []
        for row in rows:
            cells = row.find_all('td')
            cell_texts = [cell.get_text(strip=True) for cell in cells]
            data.append(cell_texts)
        return data
    return []

def click_next_button(driver):
    """다음 페이지 버튼 클릭 함수"""
    try:
        next_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.table-forward'))
        )
        next_button.click()
        time.sleep(2)  # 페이지 로딩 대기
    except Exception as e:
        print(f"Error clicking next button: {e}")

def process_page_range(start_page, end_page):
    """주어진 페이지 범위에서 데이터 추출 및 저장"""
    
    all_data = []
    for page in range(start_page, end_page + 1):
        
        # 데이터 추출
        page_data = get_page_data(driver)
        all_data.extend(page_data)
        if page_data:
            for row in page_data:
                print(row)
        # 페이지 네비게이션
        if page < end_page:
            try:
                page_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.ID, f"page{page + 1}"))
                )
                page_button.click()
                time.sleep(2)  # 페이지 로딩 대기
            except Exception as e:
                print(f"Error navigating to page {page}: {e}")

    return all_data

def main():
    all_data = []
    
    total_pages = 716
    page_range = 10

    current_page = 1
    while current_page <= total_pages:
        end_page = min(current_page + page_range - 1, total_pages)
        all_data.extend(process_page_range(current_page, end_page))
        
        # "다음" 버튼 클릭하여 다음 페이지 범위로 이동
        if end_page < total_pages:
            click_next_button(driver)
        
        # 현재 페이지 업데이트
        current_page = end_page + 1

    # 데이터프레임으로 변환
    # 데이터의 열 수를 동적으로 결정하기 위해 첫 번째 데이터의 열 수를 확인합니다.
    if all_data:
        num_columns = len(all_data[0])  # 첫 번째 항목의 열 수를 가져옵니다.
        columns = [f"Column{i+1}" for i in range(num_columns)]  # 열 이름을 설정합니다.
        df = pd.DataFrame(all_data, columns=columns)
    else:
        df = pd.DataFrame()  # 데이터가 없는 경우 빈 데이터프레임을 생성합니다.

    # DataFrame을 CSV 파일로 저장 (utf-8-sig 인코딩 사용)
    df.to_csv('crawled_data2.csv', index=False, encoding='utf-8-sig')

    print("Data saved to crawled_data2.csv")
    driver.quit()

if __name__ == "__main__":
    main()
